In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [ ]:
from langchain.agents import create_react_agent, AgentExecutor


In [ ]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

In [ ]:
search_tool = TavilySearchResults(
    max_results=2,
)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    # Placeholder function to simulate fetching weather data
    """Fetch weather data for a given city."""

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()
    if "current" not in data:
        return f"Could not retrieve weather data for {city}."
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )

In [ ]:
result=search_tool.invoke("latest news on AI research")
result

In [ ]:
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
# 3. Iterate through the server's live registry
print("Available Gemini Models for your API Key:\n" + "-"*40)
for model in genai.list_models():
    # Filter for models that specifically support text generation
    if 'generateContent' in model.supported_generation_methods:
        print(f"Model String: {model.name}")
        print(f"Description: {model.description}\n")

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0,
    google_api_key=os.environ.get("GOOGLE_API_KEY")
)

In [ ]:
response=llm.invoke("What year is it?")
response

In [ ]:
prompt=hub.pull("hwchase17/react")

In [ ]:
tools = [search_tool,get_weather_data]

In [ ]:
agent=create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

In [ ]:
response=agent_executor.invoke({
    "input": (
        "What is the capital of India."
        "and what is the current weather there?"
    )
})

In [ ]:
print(response["output"])